<a href="https://colab.research.google.com/github/DarshiniMahesh/AI-Practice/blob/main/Digit_recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
import numpy as np
import tensorflow as tf
import cv2
import base64, json
from google.colab import output
from IPython.display import HTML, display

In [34]:
mnist = tf.keras.datasets.mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalize properly
x_train = x_train / 255.0
x_test = x_test / 255.0

In [35]:
model = tf.keras.models.Sequential()

model.add(tf.keras.layers.Reshape((28,28,1), input_shape=(28,28)))

model.add(tf.keras.layers.Conv2D(32, (3,3), activation='relu'))
model.add(tf.keras.layers.MaxPooling2D())

model.add(tf.keras.layers.Conv2D(64, (3,3), activation='relu'))
model.add(tf.keras.layers.MaxPooling2D())

model.add(tf.keras.layers.Flatten())

model.add(tf.keras.layers.Dense(128, activation='relu'))
model.add(tf.keras.layers.Dense(10, activation='softmax'))

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/reshape.py:38: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ reshape (Reshape)               │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       204,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 225,034 (879.04 KB)

 Trainable params: 225,034 (879.04 KB)

 Non-trainable params: 0 (0.00 B)

In [36]:
model.fit(x_train, y_train, epochs=3, validation_data=(x_test, y_test))

Epoch 1/3
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 87s 45ms/step - accuracy: 0.9632 - loss: 0.1225 - val_accuracy: 0.9779 - val_loss: 0.0691
Epoch 2/3
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 84s 45ms/step - accuracy: 0.9871 - loss: 0.0412 - val_accuracy: 0.9877 - val_loss: 0.0364
Epoch 3/3
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 85s 45ms/step - accuracy: 0.9913 - loss: 0.0284 - val_accuracy: 0.9896 - val_loss: 0.0322


In [37]:
loss, acc = model.evaluate(x_test, y_test)
print("Accuracy:", acc)

313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 18ms/step - accuracy: 0.9896 - loss: 0.0322
Accuracy: 0.9896000027656555


In [39]:
def preprocess(img):
    img = cv2.resize(img, (28,28))

    # invert
    img = cv2.bitwise_not(img)

    # blur
    img = cv2.GaussianBlur(img, (5,5), 0)

    # threshold
    _, img = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # center digit
    coords = np.column_stack(np.where(img > 0))
    if len(coords) > 0:
        x, y, w, h = cv2.boundingRect(coords)
        img = img[y:y+h, x:x+w]
        img = cv2.resize(img, (28,28))

    # normalize
    img = img / 255.0

    return img.reshape(1,28,28,1)

In [40]:
!pip install gradio -q

import gradio as gr

def predict_digit(img):
    # Resize to 28x28
    img = img.resize((28, 28))

    # Convert to grayscale
    img = img.convert('L')

    img = np.array(img)

    # Invert colors (IMPORTANT)
    img = 255 - img

    # Normalize
    img = img / 255.0

    # Reshape
    img = img.reshape(1, 28, 28)

    # Predict
    prediction = model.predict(img)[0]

    return {str(i): float(prediction[i]) for i in range(10)}

app = gr.Interface(
    fn=predict_digit,
    inputs=gr.Image(type="pil", label="Draw or Upload Digit"),
    outputs=gr.Label(num_top_classes=3),
    title="Handwritten Digit Recognition",
    description="Draw or upload a digit (0-9) and the model will predict it with confidence"
)

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2c11cf12f9a243b6db.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [41]:
def predict_digit(img_b64):
    img_bytes = base64.b64decode(img_b64)
    img_array = np.frombuffer(img_bytes, dtype=np.uint8)
    img = cv2.imdecode(img_array, cv2.IMREAD_GRAYSCALE)

    img = preprocess(img)

    prediction = model.predict(img, verbose=0)[0]

    return json.dumps(prediction.tolist())

output.register_callback('predict_digit', predict_digit)

print("✅ MODEL READY")

✅ MODEL READY


HTML UI


In [56]:
import json, base64
import numpy as np
import cv2
import tensorflow as tf
from google.colab import output

def predict_digit(img_b64):
    try:
        img_bytes = base64.b64decode(img_b64)
        img_array = np.frombuffer(img_bytes, dtype=np.uint8)
        img = cv2.imdecode(img_array, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (28, 28))
        img = img / 255.0
        img = img.reshape(1, 28, 28).astype(np.float32)
        img_tensor = tf.constant(img)
        prediction = model(img_tensor, training=False).numpy()[0]

        # ✅ Return plain Python list — NOT json.dumps
        return prediction.tolist()

    except Exception as e:
        import traceback
        traceback.print_exc()
        return [0.0] * 10

output.register_callback('predict_digit', predict_digit)
print("✅ Callback registered!")

✅ Callback registered!


In [59]:
from IPython.display import display, HTML

ui_html = """
<!DOCTYPE html>
<html>
<head>
<style>
body { background:#050a0f; color:white; text-align:center; font-family:Arial; }
canvas { border:2px solid cyan; background:black; cursor:crosshair; }
button { margin:10px; padding:10px 20px; font-size:16px; background:#00f5c4; border:none; border-radius:4px; cursor:pointer; font-weight:bold; }
#result { color:#00f5c4; font-size:28px; }
</style>
</head>
<body>

<h2>Digit Recognition (Draw)</h2>
<canvas id="c" width="280" height="280"></canvas><br>
<button onclick="predict()">Predict</button>
<button onclick="clearCanvas()" style="background:#444;color:white;">Clear</button>
<h1 id="result">Draw a digit</h1>

<script>
const canvas = document.getElementById("c");
const ctx = canvas.getContext("2d");

ctx.fillStyle = "black";
ctx.fillRect(0, 0, 280, 280);
ctx.strokeStyle = "white";
ctx.lineWidth = 18;
ctx.lineCap = "round";
ctx.lineJoin = "round";

let drawing = false, lastX = 0, lastY = 0;

function getPos(e) {
  const r = canvas.getBoundingClientRect();
  return { x: e.clientX - r.left, y: e.clientY - r.top };
}

canvas.onmousedown = e => {
  drawing = true;
  const p = getPos(e);
  lastX = p.x; lastY = p.y;
};

canvas.onmousemove = e => {
  if (!drawing) return;
  const p = getPos(e);
  ctx.beginPath();
  ctx.moveTo(lastX, lastY);
  ctx.lineTo(p.x, p.y);
  ctx.stroke();
  lastX = p.x; lastY = p.y;
};

canvas.onmouseup = () => drawing = false;
canvas.onmouseleave = () => drawing = false;

function clearCanvas() {
  ctx.fillStyle = "black";
  ctx.fillRect(0, 0, 280, 280);
  document.getElementById("result").innerText = "Draw a digit";
}

async function predict() {
  document.getElementById("result").innerText = "Predicting...";
  try {
    const b64 = canvas.toDataURL("image/png").split(',')[1];

    const res = await google.colab.kernel.invokeFunction(
      'predict_digit', [b64], {}
    );

    // ✅ Fixed — data is in text/plain
    const probs = JSON.parse(res.data['text/plain']);

    const pred = probs.indexOf(Math.max(...probs));
    const conf = (Math.max(...probs) * 100).toFixed(2);

    document.getElementById("result").innerText =
      "Predicted: " + pred + " (" + conf + "%)";

  } catch(err) {
    document.getElementById("result").innerText = "Error: " + err;
  }
}
</script>
</body>
</html>
"""

display(HTML(ui_html))